#Network Intrusion Detection System

## Data Load And Package Import

In [ ]:
# Data Exploration and Preprocessing
import pandas as pd
import numpy as np
import missingno as mno
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearnex import patch_sklearn
pd.set_option('display.max_columns', None)
patch_sklearn()

# Feature Selection
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import Lasso

# Model Selection
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

# Pipeline
from sklearn.pipeline import Pipeline

# Model Evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv(r'/content/drive/MyDrive/MachineLearningCSV/MachineLearningCVE/Network intrustion Data.csv')

## Data Preprocessing


In [ ]:

# The words to be deleted
words_to_delete = ['Min', 'Max', 'Mean', 'Std']

# Drop all columns that contain any of the words
df = df.drop(columns=df.columns[df.columns.str.contains('|'.join(words_to_delete))])

# Print the DataFrame
print("Cleaned DataFrame:")
print(df.head())

In [ ]:
# Check for missing information in the DataFrame
missing_info = df.isnull().sum()
print("Missing Information in DataFrame:")
print(missing_info)

In [ ]:
# Count missing 'Flow Bytes/s' information for specific labels
missing_flow_bytes = df[' Label'].loc[df['Flow Bytes/s'].isnull() == True].value_counts()
print("\nMissing Flow Bytes/s Information for Specific Labels:")
print(missing_flow_bytes)

#Explanation for the results obtained:
# The code checks for missing information in the DataFrame and counts the missing 'Flow Bytes/s'
#information for specific labels. It also mentions that there's missing information for 'DoS Hulk' and 'Benign'.
#A decision has been made to impute 'Mode' values for respective labels.
#Additionally, it notes that 'Flow Packets' has 'inf' values

In [ ]:


categories = df.iloc[:, -1].unique()  # Get unique categories from the last column

# Create a dictionary to store descriptions
category_descriptions = {}

# Loop through each category
for category in categories:
    category_df = df[df.iloc[:, -1] == category]  # Filter the DataFrame for the current category
    description = category_df.describe()  # Calculate the descriptive statistics
    category_descriptions[category] = description  # Store the description in the dictionary

In [ ]:

# Drop rows with any NaN values
df = df.dropna()

# Drop rows with 'inf' values in any column
df = df[~df.applymap(lambda x: isinstance(x, (float, np.float64)) and np.isinf(x)).any(axis=1)]

# Drop rows with any NaN values again
df = df[~df.isna().any(axis=1)]

# The code drops rows containing NaN or 'inf' values in any column of the DataFrame.
# It first drops rows with NaN values and then drops rows with 'inf' values in any column.

In [ ]:
# Clean column names by removing leading spaces
df.columns = df.columns.str.lstrip(' ')

In [ ]:
# Remove special characters from ' Label' column
df[' Label'] = df[' Label'].str.replace('�', '')
df[' Label'] = df[' Label'].str.replace('-', '')

##Feature Selection

In [ ]:
# Separate features (x) and target variable (y)
x = df.drop(columns=' Label')
y = df[' Label'].astype(str)

# Reset the index of the feature DataFrame
x.reset_index(drop=True, inplace=True)  # Added inplace=True to modify x in place


###Random Forest Feature Selection

In [ ]:
#from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest classifier model
model = RandomForestClassifier()

# Fit the model with the features (x) and target variable (y)
model.fit(x, y)

# Calculate feature importances
feature_scores = model.feature_importances_
feature_names = x.columns

# Create a DataFrame to store feature importances
feature_importance_df = pd.DataFrame({'Features': feature_names, 'Importance': feature_scores})

# Sort the features by importance (descending order)
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

# Plot and display feature importance scores
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 8))
plt.bar(feature_importance_df['Features'], feature_importance_df['Importance'])
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.title('Feature Importance Scores (Descending Order)')
plt.xticks(rotation=90)  # Rotate x-axis labels for better visibility
plt.tight_layout()
plt.show()

# Display the sorted feature importance DataFrame
feature_importance_df

###Lasso Regression

In [ ]:

#Calling feature names
features = x.columns

#Encoder
encoder = LabelEncoder()
y_enc = encoder.fit_transform(y)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(x, y_enc, test_size=0.33, random_state=42)

#Creating a pipeline to call
pipeline = Pipeline([
                     ('scaler',StandardScaler()),
                     ('model',Lasso())
])

#Initiating a grid search loop to run the pipeline to find optimal Alpha
search = GridSearchCV(pipeline,
                      {'model__alpha':np.arange(0.1,10,0.1)},
                      cv = 5, scoring="neg_mean_squared_error",verbose=3
                      )

#intitating search
search.fit(X_train,y_train)

search.best_params_

coefficients = search.best_estimator_.named_steps['model'].coef_

importance = np.abs(coefficients)

np.array(features)[importance > 0]

In [ ]:
#Lasso Plot

# Create a dictionary with feature names and their importance scores
imp_dict = {'Columns': features, 'Importance': importance}

# Convert the dictionary to a DataFrame
imp_df = pd.DataFrame.from_dict(imp_dict)

# Filter out features with zero importance
imp_df = imp_df.loc[imp_df['Importance'] != 0]

# Create a bar plot to visualize feature importance
plt.figure(figsize=(10, 6))
ax = imp_df.plot(kind='bar', x='Columns', y='Importance')

# Set plot labels and title
plt.title('Lasso Importance Scores')
plt.xlabel('Columns')
plt.ylabel('Importance Score')

# Add labels to the bars with rounded importance scores
ax.bar_label(ax.containers[0], label_type='edge', labels=imp_df['Importance'].round(2))

# Show the plot
plt.show()

###Univariate Feature Selection

In [ ]:


# Perform univariate feature selection using the f_classif test
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(x, y)

# Get the selected features
features = selector.get_support()

# Get the column names of the features
names = x.columns

# Get the scores of the features
scores = selector.scores_

# Create a DataFrame with the column names, scores, and true/false marks
df_scores = pd.DataFrame({"Features": names, "Scores": scores, "Selected": features})

# Print the DataFrame
display(df_scores.sort_values(by = 'Selected', ascending = False))

###Correlation Matrix

In [ ]:

# Drop specified columns
columns_to_drop = ['Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate']
df_cleaned = df.drop(columns=columns_to_drop)

# Correlation Heatmap

plt.figure(figsize=(20, 8))
sns.heatmap(df_cleaned.corr())
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Finding Columns with Only One Value

def find_columns_with_one_value(df):
    columns_with_one_value = []
    for col in df.columns:
        if df[col].nunique() == 1:
            columns_with_one_value.append((col, df[col].unique()[0]))
    return columns_with_one_value

# Find the columns with only one value
columns_with_one_value = find_columns_with_one_value(df_cleaned)

# Print the columns with only one value
for col, value in columns_with_one_value:
    print(f"{col}: {value}")

In [ ]:
# Correlation Analysis

# Drop columns with low correlation
columns_to_drop = [' Bwd PSH Flags', ' Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
                   ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk',
                   ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
corr = df_cleaned.corr().drop(columns=columns_to_drop)

# Compute correlation matrix
df_corr = df_cleaned.copy()
df_corr.drop(columns=columns_to_drop, inplace=True)
df_corr_matrix = df_corr.corr(method='pearson')

# Filter upper triangle of correlation matrix
upper_corr_matrix = df_corr_matrix.where(np.triu(np.ones(df_corr_matrix.shape), k=1).astype(np.bool))

# Convert to 1-D series and drop Null values
unique_corr_pairs = upper_corr_matrix.unstack().dropna()

# Sort correlation pairs
sorted_corr_pairs = unique_corr_pairs.sort_values(ascending=False)

# Print sorted correlation pairs
print(sorted_corr_pairs)

# Save the correlation matrix to a CSV file
corr_matrix.to_csv('corr_matrix.csv')

##Feature Elimination

In [ ]:
# Sort DataFrame columns in ascending order
sorted_columns = df.columns.sort_values(ascending=True)

# Define a list of columns to remove
rem_cols = ['Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd PSH Flags',
            'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bulk Rate', 'Fwd Avg Packets/Bulk',
            'Fwd URG Flags', 'RST Flag Count', 'Bwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'ECE Flag Count',
            'Fwd Header Length.1']


In [ ]:
# Create a new DataFrame by dropping the specified columns
df_new = df.drop(columns=rem_cols)

# Save the new DataFrame to a CSV file
df_new.to_csv(r'/content/drive/MyDrive/MachineLearningCVE/Preprocessed_df.csv')


## Model Creation

In [ ]:
!pip install scikit-learn-intelex

### Train-Test Split

In [ ]:
# Define the target variable
target = df['Label']

# Import the necessary libraries
from sklearn.model_selection import train_test_split

# Separate the features and target variable
X = df.drop(columns='Label')
y = df['Label']

# Split the data into training and testing sets (stratified sampling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify=y, random_state=42)


###Random Forest

In [ ]:


# Create a Random Forest classifier
rf = RandomForestClassifier()

# Fit the classifier to the training data
rf.fit(X_train, y_train)

# Make predictions on the test data
y_pred_rf = rf.predict(X_test)

# Print a classification report
print(classification_report(y_test, y_pred_rf, output_dict=True))



###Decision Tree

In [ ]:

# Create a Decision Tree classifier
dt = DecisionTreeClassifier()

# Fit the classifier on the training data
dt.fit(X_train, y_train)

# Make predictions on the test data
y_pred_dt = dt.predict(X_test)

# Print the classification report
print(classification_report(y_test, y_pred_dt))


###SVM

In [ ]:

# Create an SVM classifier (uncomment the lines below)
svc = SVC()
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)

# Print the classification report (uncomment the line below)
print(classification_report(y_test, y_pred_svc))


###Logistic Regression

In [ ]:

# Create a Logistic Regression classifier with verbosity (verbose=1)
lr = LogisticRegression(verbose=1)

# Fit the classifier on the training data
lr.fit(X_train, y_train)

# Make predictions on the test data
y_pred_lr = lr.predict(X_test)

# Print the classification report
print(classification_report(y_test, y_pred_lr))


###XGBoost

In [ ]:

# Create an XGBoost classifier
xgb = XGBClassifier()

# Encode the target variable
enc = LabelEncoder()
y_enc = enc.fit_transform(y)

# Split the data into training and testing sets (stratified sampling)
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X, y_enc, test_size=0.33, stratify=y_enc, random_state=42)

# Fit the XGBoost classifier on the training data
xgb.fit(X_train_xgb, y_train_xgb)

# Make predictions on the test data
y_pred_xgb = xgb.predict(X_test)

# Print the classification report
print(classification_report(y_test_xgb, y_pred_xgb))


##Hyperparameter Optimization

In [ ]:
# Random Forest

# Define the hyperparameters to tune
parameters = {
    "n_estimators": [50, 100, 150, 200, 250, 500],
    "max_features": ["sqrt", "log2"],
}

# Create a GridSearchCV object with RandomForestClassifier
clf = GridSearchCV(RandomForestClassifier(), parameters, cv=5)

# Fit the model to the training data
clf.fit(X_train, y_train)

# Print the best hyperparameters found by the grid search
print("Best Hyperparameters:")
print(clf.best_params_)

rnd_forest = RandomForestClassifier(n_estimators = 250, max_features= 'sqrt')

In [ ]:
# Decision Tree

# Define the hyperparameters to tune
parameters = {
    "criterion": ["gini", "entropy", "logloss"],
    "splitter": ["best", "random"]
}

# Create a GridSearchCV object with DecisionTreeClassifier
clf2 = GridSearchCV(DecisionTreeClassifier(), parameters, cv=5)

# Fit the model to the training data
clf2.fit(X_train, y_train)

# Print the best hyperparameters found by the grid search
print("Best Hyperparameters:")
print(clf2.best_params_)

dc_tree = DecisionTreeClassifier(criterion = 'entropy', splitter = 'best')


##Voting Classifier  - Ensemble

In [ ]:

# Create a VotingClassifier for hard voting
vc1 = VotingClassifier(estimators=[('dt', dc_tree), ('rf', rnd_forest), ('xgb', xgb)], voting='hard')

# Fit VotingClassifier on the training data
vc1.fit(X_train, y_train)

# Make predictions using VotingClassifier
y_pred_vc1 = vc1.predict(X_test)

# Print classification reports
print("VC1 Classification Report:")
print(classification_report(y_test, y_pred_vc1))

# Plot heatmaps for classification reports
plt.figure(figsize=(12, 6))
sns.heatmap(pd.DataFrame(classification_report(y_test, y_pred_vc1, output_dict=True)).iloc[:-1, :].T, annot=True)
plt.title('VC1 Classification Report')
plt.show()


In [ ]:
# Create a VotingClassifier for soft voting
vc2 = VotingClassifier(estimators=[('dt', dc_tree), ('rf', rnd_forest), ('xgb', xgb)], voting='soft')

# Fit VotingClassifier on the training data
vc2.fit(X_train, y_train)

# Make predictions using VotingClassifier
y_pred_vc2 = vc2.predict(X_test)

# Print classification reports
print("\nVC2 Classification Report:")
print(classification_report(y_test, y_pred_vc2))

# Plot heatmaps for classification reports
plt.figure(figsize=(12, 6))
sns.heatmap(pd.DataFrame(classification_report(y_test, y_pred_vc2, output_dict=True)).iloc[:-1, :].T, annot=True)
plt.title('VC2 Classification Report')
plt.show()

In [ ]:
# Create a DataFrame y_test with the predicted labels
y_test['pred_label'] = y_pred

# Create a DataFrame test_y
test_y = pd.DataFrame(y_test)

# Remove the 'pred_label' column from test_y
test_y.drop('pred_label', axis=1, inplace=True)

# Add the 'pred_label' column back to test_y
test_y['pred_label'] = y_pred

# Identify rows where the true label does not match the predicted label
mismatched_rows = test_y.loc[test_y['Label'] != test_y['pred_label']]

# Merge X_test with the DataFrame containing true and predicted labels
clf_df = X_test.merge(test_y[['Label', 'pred_label']], how='left', left_index=True, right_index=True)

# Drop the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in clf_df.columns:
    clf_df.drop(columns='Unnamed: 0', inplace=True)

# Save the resulting DataFrame to a CSV file
clf_df.to_csv('/content/drive/MyDrive/MachineLearningCVE/CLF_Output.csv', index=False)


##Unsupervised Model


In [ ]:
# Filter data for the 'BENIGN' class
df_x = df[df['pred_label'] == 'BENIGN']

# Drop the 'pred_label' column to prepare data for clustering
X = df_x.drop(columns='pred_label')

# Standardize the data (important for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Perform PCA for dimensionality reduction
pca = PCA(n_components=2)  # You can choose the number of components
X_pca = pca.fit_transform(X_scaled)

# Find the optimal number of clusters using the Elbow method
model = KMeans(init='k-means++')  # Using KMeans++ initialization
visualizer = KElbowVisualizer(model, k=(2, 10))
visualizer.fit(X_pca)  # Fit the data to the visualizer
visualizer.show()

# Choose the number of clusters (e.g., 4) based on the Elbow plot
n_clusters = 4

# Fit KMeans with the chosen number of clusters
kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42)
kmeans.fit(X_pca)

# Get cluster labels for each data point
labels = kmeans.labels_

# Calculate silhouette score
silhouette_avg = silhouette_score(X_pca, labels)
print(f"Silhouette Score for {n_clusters} clusters:", silhouette_avg)